In [1]:
!pip install -q markdownify beautifulsoup4 requests

In [3]:
"""
Genera _indice.md en la carpeta de la KB a partir del JSON de doctorados UPV.
Ejecutar en Google Colab después de montar el Drive.
"""

import json
import os
from google.colab import drive

drive.mount('/content/drive')


JSON_URL = '/content/drive/MyDrive/TFG Teleco/JSONs/doctorados_upv.json'
PATH_INDICE = '/content/drive/MyDrive/TFG Teleco/UPV_Doctorados_KB/_indice.md'


with open(JSON_URL, encoding='utf-8') as f:
    datos = json.load(f)


tipos_idx = {
    t['tipo']: t['texto']
    for t in datos.get('tipos', [])
}


lineas = [
    '# Índice de Doctorados UPV',
    '',
    '| Acrónimo | Título | Tipo |',
    '|----------|--------|------|',
]


for d in sorted(datos['titulaciones'], key=lambda x: x['nom']):

    acro = d['acro']
    nom = d['nom']

    tipos = ', '.join(
        tipos_idx.get(t, t)
        for t in d.get('tipo', [])
    )

    url = f"https://www.upv.es{d['url']}"

    lineas.append(
        f'| [{acro}]({url}) | {nom} | {tipos} |'
    )


os.makedirs(
    os.path.dirname(PATH_INDICE),
    exist_ok=True
)


with open(
    PATH_INDICE,
    'w',
    encoding='utf-8'
) as f:
    f.write('\n'.join(lineas))


print(
    f"✅ Índice generado: {len(datos['titulaciones'])} doctorados → {PATH_INDICE}"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Índice generado: 32 doctorados → /content/drive/MyDrive/TFG Teleco/UPV_Doctorados_KB/_indice.md


In [8]:
"""
UPV Doctorados Knowledge Base Extractor
=======================================

Adaptación del extractor de másteres UPV.

Entrada:
    JSON de doctorados generado:
    /content/drive/MyDrive/TFG Teleco/JSONs/doctorados_upv.json

Salida:
    /content/drive/MyDrive/TFG Teleco/UPV_Doctorados_KB/*.md
"""


# ─────────────────────────────────────────────
# CELDA 1 — Instalación
# ─────────────────────────────────────────────

#!pip install -q markdownify beautifulsoup4 requests


# ─────────────────────────────────────────────
# CELDA 2 — Imports y configuración
# ─────────────────────────────────────────────

import os, re, time, pickle, requests, markdownify, json

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from google.colab import drive


PATH_DRIVE = '/content/drive/MyDrive/TFG Teleco/UPV_Doctorados_KB'


JSON_URL = '/content/drive/MyDrive/TFG Teleco/JSONs/doctorados_upv.json'


BASE_URL = 'https://www.upv.es'


ARCHIVO_ESTADO = os.path.join(
    PATH_DRIVE,
    '_estado.pkl'
)


PAUSA = 0.4


HEADERS = {
    'User-Agent':
    'Mozilla/5.0 (compatible; UPV-KB-Bot/2.0)'
}


# Secciones del programa de doctorado
SECCIONES = {
    'Inicio': '',
    'Admisión': 'admision/',
}


drive.mount('/content/drive')


os.makedirs(
    PATH_DRIVE,
    exist_ok=True
)


print(f"✅ Drive montado → {PATH_DRIVE}")




# ─────────────────────────────────────────────
# CELDA 4 — Extracción de contenido
# ─────────────────────────────────────────────

# ── Secciones de ruido a eliminar de Inicio ──────────────────────────────────
# Headings que marcan el inicio de bloques irrelevantes para una KB
HEADINGS_RUIDO = {
    'Conoce el máster a fondo', 'Galería de imágenes',
    'Actualidad del máster', 'Mantente al día',
    'Normativa general', 'Otros enlaces de interés',
    'Conoce el master a fondo',
}

# Texto de bloques promocionales estáticos (iguales en todos los másteres)
TEXTO_PROMO = re.compile(
    r'Desde 1991 hemos gestionado|Hemos creado 5500|hemos tramitado \d',
    re.I
)


def _limpiar_ruido_inicio(main):
    """Elimina in-place secciones ruidosas del árbol BeautifulSoup de Inicio."""

    # 1. Eliminar por heading: busca h2/h3 con texto de ruido y elimina su sección
    for heading in main.find_all(['h1', 'h2', 'h3', 'h4']):
        texto_h = heading.get_text(strip=True)
        if any(r.lower() in texto_h.lower() for r in HEADINGS_RUIDO):
            # Intentar eliminar el bloque contenedor (section / wp-block-group)
            padre = heading.find_parent(
                lambda t: t.name in ('section', 'div', 'aside', 'article')
                and t != main
            )
            if padre:
                padre.decompose()
            else:
                # Sin bloque contenedor: eliminar heading + hermanos siguientes
                for sib in list(heading.find_next_siblings()):
                    sib.decompose()
                heading.decompose()

    # 2. Eliminar noticias individuales (contienen "Lee más:")
    for a in main.find_all('a', string=re.compile(r'Lee más', re.I)):
        contenedor = a.find_parent(['li', 'article', 'div'])
        if contenedor:
            contenedor.decompose()

    # 3. Eliminar bloques promocionales estáticos
    for nodo in main.find_all(string=TEXTO_PROMO):
        bloque = nodo.find_parent(['div', 'section', 'p', 'h3'])
        if bloque:
            bloque.decompose()

    # 4. Eliminar galerías e imágenes sueltas (no aportan a la KB)
    for sel in ['.wp-block-gallery', 'figure.wp-block-image', '[class*="galeria"]']:
        for tag in main.select(sel):
            tag.decompose()


def extraer_hero_metadata(soup: BeautifulSoup) -> str:
    """
    Extrae los datos del banner hero que aparece en la cabecera de cada página
    de máster: acrónimo, idioma de impartición, requisito lingüístico,
    modalidad, campus, créditos, tipo de título.

    Estos datos están ANTES de <main>, por lo que hay que leerlos antes de
    que limpiar_html elimine la cabecera.
    """
    hero = soup.find(class_='seccion01hero')
    if not hero:
        return ''

    lineas = ['### Datos del programa\n']

    # Fila superior: "Título oficial", "60 créditos", etc.
    for col in hero.find_all(class_='columna_arriba'):
        t = col.get_text(' ', strip=True)
        if t:
            lineas.append(f"- {t}")

    lineas.append('')

    # Fila inferior: ACRÓNIMO → valor, IDIOMA → valor, etc.
    for col in hero.find_all(class_='columna_abajo'):
        h = col.find(['h3', 'h4'])
        p = col.find('p')
        if h and p:
            clave = h.get_text(strip=True)
            valor = p.get_text(strip=True)
            if clave and valor:
                lineas.append(f"**{clave}**: {valor}")

    return '\n'.join(lineas) if len(lineas) > 2 else ''


def seguir_iframe_oracle(soup: BeautifulSoup) -> str:
    """
    Detecta iframes que apuntan a páginas Oracle/UPV (pls/oalu/...)
    y descarga su contenido.

    Estas páginas contienen el listado real de asignaturas, competencias,
    profesorado y contenidos de "En detalle".
    """
    # Buscar en el contenedor upv_query o cualquier iframe Oracle
    iframe = (
        (soup.find(class_='upv_query') or soup).find(
            'iframe', src=re.compile(r'pls/oalu|oalu/sic_', re.I)
        )
    )
    if not iframe or not iframe.get('src'):
        return ''

    src = iframe['src']
    if src.startswith('//'):
        src = 'https:' + src
    elif src.startswith('/'):
        src = BASE_URL + src

    print(f"      ↪ iframe Oracle: {src[:90]}...")
    resp = get(src)
    if not resp:
        return ''

    isoup = BeautifulSoup(resp.text, 'html.parser')
    for tag in isoup(['script', 'style', 'noscript', 'link', 'meta']):
        tag.decompose()

    body = isoup.find('body') or isoup
    md = markdownify.markdownify(
        str(body), heading_style='ATX', strip=['a', 'img']
    )
    return re.sub(r'\n{3,}', '\n\n', md).strip()


def limpiar_html_pagina(soup: BeautifulSoup, es_inicio: bool = False) -> str:
    """
    Limpia la página WordPress del máster y devuelve el contenido útil
    en Markdown.

    Para la sección Inicio aplica además la limpieza de ruido específica.
    """
    # ── 1. Tags estructurales globales ────────────────────────────────────
    for tag in soup(['script', 'style', 'noscript', 'link', 'meta',
                     'header', 'footer', 'nav']):
        tag.decompose()

    # ── 2. Selectores CSS: cabecera global UPV, menús, pie ────────────────
    for sel in [
        '.master-global-header', '.master-global-header-secondary',
        '#masthead-container', '#masthead', '#colophon', '#site-navigation',
        '.global-menu', '.bg-overlay', '.breadcrumb', '.social-sharer',
        '.cookies-banner', '.menu-lateral', '.accessible-megamenu',
        '.boton_policonsulta', '.footer-area', '.footer-bottom-bar',
    ]:
        for tag in soup.select(sel):
            tag.decompose()

    # ── 3. Banner "Conoce la Universitat" ─────────────────────────────────
    # Comprobamos SOLO clase e id del propio tag, nunca get_text().
    # get_text() incluye el texto de todos los descendientes: si un <div> grande
    # contiene en algún hijo esa cadena, el nodo entero —con su contenido útil—
    # quedaría eliminado. Eso es lo que borraba "En detalle".
    for tag in soup.find_all(['div', 'section', 'aside', 'article']):
        try:
            clases = ' '.join(tag.get('class', []))
            id_tag = tag.get('id', '')
            if ('conoce-la-universitat' in clases + id_tag
                    or 'bloque-conoce' in clases + id_tag):
                tag.decompose()
        except Exception:
            pass

    # ── 4. Localizar contenido principal ──────────────────────────────────
    main = (
        soup.find('main')
        or soup.find(id='primary')
        or soup.find(id='cuerpo')
        or soup.find(class_=re.compile(r'entry-content|page-content'))
        or soup.body
    )
    if not main:
        return ''

    # ── 5. Limpieza de ruido específica para Inicio ───────────────────────
    if es_inicio:
        _limpiar_ruido_inicio(main)

    md = markdownify.markdownify(
        str(main),
        heading_style='ATX',
        strip=['a', 'img'],   # sin URLs ni imágenes en la KB
    )
    return re.sub(r'\n{3,}', '\n\n', md).strip()


def extraer_seccion(url: str, nombre: str, es_inicio: bool = False) -> str:
    """
    Descarga una sección del máster y devuelve su contenido en Markdown.

    Estrategia por sección:
      - Inicio     → hero metadata + contenido limpio sin ruido
      - En detalle, Asignaturas, Competencias, Profesorado
                   → intenta seguir el iframe Oracle; si no existe, usa HTML directo
      - Admisión   → contenido HTML directo
    """
    resp = get(url)
    if resp is None:
        return f'> ⚠️ No se pudo obtener "{nombre}" ({url})\n'

    soup = BeautifulSoup(resp.text, 'html.parser')

    partes = []

    # ── Hero banner (solo en Inicio) ──────────────────────────────────────
    if es_inicio:
        hero = extraer_hero_metadata(soup)
        if hero:
            partes.append(hero)

    # ── Contenido del iframe Oracle (si existe) ───────────────────────────
    iframe_md = seguir_iframe_oracle(soup)
    if iframe_md:
        partes.append(iframe_md)

    # ── Contenido HTML directo ────────────────────────────────────────────
    # Se usa siempre para Inicio y Admisión; para las demás solo si no hay iframe
    if es_inicio or nombre == 'Admisión' or not iframe_md:
        html_md = limpiar_html_pagina(soup, es_inicio=es_inicio)
        if html_md:
            partes.append(html_md)

    if not partes:
        return f'> ℹ️ Sección "{nombre}" sin contenido extraíble.\n'

    return '\n\n'.join(partes)


# ─────────────────────────────────────────────
# CELDA 5 — Estado (reanudación entre sesiones)
# ─────────────────────────────────────────────

def cargar_estado() -> set:
    if os.path.exists(ARCHIVO_ESTADO):
        with open(ARCHIVO_ESTADO, 'rb') as f:
            return pickle.load(f)
    return set()


def guardar_estado(procesados: set):
    with open(ARCHIVO_ESTADO, 'wb') as f:
        pickle.dump(procesados, f)


# ─────────────────────────────────────────────
# CELDA 6 — Bucle principal
# ─────────────────────────────────────────────

def construir_metadata(m, ramas_idx, campus_idx, centros_idx, tipos_idx) -> str:
    """Cabecera YAML-like con los metadatos del JSON para facilitar el filtrado en RAG."""
    ramas   = ', '.join(ramas_idx.get(r, str(r)) for r in m.get('ramas', []))
    campus  = campus_idx.get(m.get('id_campus', ''), m.get('id_campus', ''))
    centros = ', '.join(centros_idx.get(c, c) for c in m.get('centros', []))
    tipos   = ', '.join(tipos_idx.get(t, t) for t in m.get('tipo', []))
    modal   = {'1': 'Presencial', '2': 'Semipresencial', '3': 'En línea'}.get(
                  str(m.get('id_modalidad', '')), '')
    return '\n'.join([
        '---',
        f'título: "{m["nom"]}"',
        f'acrónimo: {m["acro"]}',
        f'tipo: {tipos}',
        f'campus: {campus}',
        f'modalidad: {modal}',
        f'centros: {centros}',
        f'ramas: {ramas}',
        f'url: {BASE_URL}{m["url"]}',
        '---', '',
    ])


def procesar_doctorados():
    print("📥 Descargando catálogo de doctorados...")

    # JSON local en Drive
    if JSON_URL.startswith('/content/'):
        with open(JSON_URL, 'r', encoding='utf-8') as f:
            datos = json.load(f)
    else:
        resp = get(JSON_URL)
        if resp is None:
            raise RuntimeError("No se pudo descargar el JSON de doctorados.")
        datos = resp.json()


    doctorados = datos['titulaciones']

    ramas_idx   = {r['id_rama']: r['nom'] for r in datos.get('ramas', [])}
    campus_idx  = {c['id_campus']: c['nom'] for c in datos.get('campus', [])}
    centros_idx = {c['acro']: c['nom'] for c in datos.get('centros', [])}
    tipos_idx   = {t['tipo']: t['texto'] for t in datos.get('tipos', [])}


    print(f"📚 {len(doctorados)} títulos en el catálogo.")


    procesados = cargar_estado()
    pendientes = [d for d in doctorados if d['acro'] not in procesados]

    print(
        f"🔄 {len(procesados)} ya procesados · "
        f"{len(pendientes)} pendientes."
    )


    for i, doctorado in enumerate(pendientes, 1):

        acro = doctorado['acro']
        nom  = doctorado['nom']

        url_base = urljoin(
            BASE_URL,
            doctorado['url']
        )


        print(
            f"\n[{i}/{len(pendientes)}] ── {acro}: {nom}"
        )


        partes = [
            construir_metadata(
                doctorado,
                ramas_idx,
                campus_idx,
                centros_idx,
                tipos_idx
            ),
            f"# {nom}\n",
        ]


        for nombre_sec, sufijo in SECCIONES.items():

            url_sec = (
                url_base
                if sufijo == ''
                else urljoin(url_base, sufijo)
            )

            es_inicio = (nombre_sec == 'Inicio')


            print(
                f"    → {nombre_sec}: {url_sec}"
            )


            contenido = extraer_seccion(
                url_sec,
                nombre_sec,
                es_inicio=es_inicio
            )

            partes.append(
                f"\n## {nombre_sec}\n\n{contenido}\n"
            )


        documento = '\n'.join(partes)


        nombre_archivo = (
            re.sub(r'[^\w\-]', '_', acro)
            + '.md'
        )


        ruta = os.path.join(
            PATH_DRIVE,
            nombre_archivo
        )


        with open(
            ruta,
            'w',
            encoding='utf-8'
        ) as f:
            f.write(documento)


        procesados.add(acro)


        if i % 5 == 0:
            guardar_estado(procesados)
            print(
                f"    💾 Checkpoint "
                f"({len(procesados)} procesados)"
            )


    guardar_estado(procesados)


    total = len(
        [
            f for f in os.listdir(PATH_DRIVE)
            if f.endswith('.md')
        ]
    )


    print(
        f"\n✅ Completado. "
        f"Archivos .md en Drive: {total}"
    )

    print(
        f"   Ruta: {PATH_DRIVE}"
    )


# ── Ejecutar ───────────────────────────────────────────────────────────────────
procesar_doctorados()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive montado → /content/drive/MyDrive/TFG Teleco/UPV_Doctorados_KB
📥 Descargando catálogo de doctorados...
📚 32 títulos en el catálogo.
🔄 0 ya procesados · 32 pendientes.

[1/32] ── Programa_de_doctorado_en_Recursos_y_Tecnologías_Agrícolas: Programa de doctorado en Recursos y Tecnologías Agrícolas
    → Inicio: https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-recursos-y-tecnologias-agricolas/
    → Admisión: https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-recursos-y-tecnologias-agricolas/admision/
⚠️ https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-recursos-y-tecnologias-agricolas/admision/: 404 Client Error: Not Found for url: https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-recursos-y-tecnologias-agricolas/admision/

[2/32] ── Programa_de_doctorado_en_Ciencia_y_Tecnología_de_la_Producción_A